# Market data, end to end

The demonstration of Phase 1 on your own data: the source guide and the catalog status, a year of continuous 5-minute bars with their rolls, one real day resampled from 1-minute bars and compared with the vendor's bars, and a dataset you imported through a preset with its quality report. `docs/data-guide.md` explains each part.

Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and changes nothing in it. It needs 1-minute bars of at least one product and uses MES's when your store holds them; a cell whose data your store lacks says so and skips. Above each cell is what it should print on this PC; prices print in points, and times in the time zone of the product's sessions.

## Setup

Imports, the folders FEFAT reads, and `readable`, which shows bars the way a chart does.

Expected: the path of your data folder, `%LOCALAPPDATA%\FEFAT\data` unless you set `FEFAT_DATA_DIR`.

In [ ]:
import os
import time
from datetime import timedelta

import polars as pl
from IPython.display import Markdown, display

import fefat as ff
from fefat.config import resolve_paths
from fefat.market.bars_check import ComparisonStatus, verify_bars
from fefat.market.importers.profiles import preset_profiles
from fefat.market.saved_instruments import instrument_table

paths = resolve_paths(os.environ)
store = ff.market.Store(paths.data_dir)
instruments = instrument_table(paths.data_dir)
PRICES = ("open", "high", "low", "close", "close_raw", "adj_offset")


def readable(bars: pl.DataFrame, root: str) -> pl.DataFrame:
    """The bars of `root` as a chart shows them: each bar's open in the time zone of the
    product's sessions, and its prices in points rather than in 1e-9 units."""
    zone = instruments.get(root).session_calendar().hours.time_zone
    opens = pl.from_epoch("ts_open", time_unit="ns").dt.replace_time_zone("UTC")
    return bars.select(
        "trade_date",
        "contract",
        opens=opens.dt.convert_time_zone(zone),
        **{price: pl.col(price) / 1e9 for price in PRICES},
        volume="volume",
    )


print(f"Data folder: {paths.data_dir}")

## The source guide

The well-known sources of market data, the licence scope their data comes under, and the presets that read their files; `fefat data sources` prints each in full, with what it offers, its cost, and its terms.

Expected: 27 sources researched on 2026-10-12; seven presets available, under Databento, Interactive Brokers, NinjaTrader 8, TradingView, Sierra Chart, and TradeStation, Binance Vision's planned, and none yet for the other sources.

In [ ]:
guide = ff.market.SOURCE_GUIDE
print(
    f"{len(guide.sources)} sources, researched on {guide.researched_on}: prices and terms "
    "change, so check a source's own pages before you rely on them."
)
sources = pl.DataFrame(
    [
        {
            "source": source.name,
            "kind": str(source.kind),
            "presets": ", ".join(f"{preset.id} ({preset.status})" for preset in source.presets)
            if source.presets
            else "none yet",
            "licence_scope": source.licence_scope,
        }
        for source in guide.sources
    ]
)
with pl.Config(tbl_rows=-1, fmt_str_lengths=120):
    display(sources)

## The catalog status

Each product of the catalog with its datasets, how much of the product's history they cover, and their quality, as `fefat data status` prints it.

Expected on this PC: MES, MNQ, MCL, and MGC complete at 1-minute fidelity from `databento`, clean but for MGC, flagged for four days without bars in 2014; MNQ also from `ninjatrader`, flagged for the sessions its export lacks. A product without data reads `missing`, and one outside the catalog `unlisted`.

In [ ]:
print(f"Catalog version {ff.market.CATALOG.version}: {', '.join(ff.market.CATALOG.roots())}.")
ff.market.product_status(store, paths.reports_dir).select(
    "root",
    "coverage",
    "provider",
    "schema",
    "fidelity",
    "contracts",
    "first_trade_date",
    "last_trade_date",
    "last_import",
    "quality",
    "checked_through",
)

## The datasets, and the one the next cells use

Every dataset `ff.market.load` reads, as `fefat data catalog` lists them, and the one the next cells use: MES's 1-minute bars when your store holds them, else the 1-minute bars with the most contracts.

Expected on this PC: five datasets, and MES from `databento`, 34 contracts, trade dates 2019-05-06 to 2026-09-23.

In [ ]:
listed = ff.market.catalog()
minute_bars = listed.filter(pl.col("schema") == "ohlcv-1m")
if minute_bars.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = minute_bars.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
print(
    f"The next cells use {ROOT} from {PROVIDER}: {chosen['contracts']} contracts, trade dates "
    f"{chosen['first_trade_date']} to {LAST}."
)
listed.select(
    "provider",
    "root",
    "schema",
    "fidelity",
    "contracts",
    "first_trade_date",
    "last_trade_date",
    "quality",
    "origin",
)

## A year of continuous 5-minute bars

The last year of the product's 5-minute bars, resampled from its 1-minute bars and spliced from each session's front contract into one forward-anchored series, timed.

Expected on this PC: about 71,000 bars of MES in a few seconds, no bar whose close is not `close_raw + adj_offset`, and the series' last five bars.

In [ ]:
FIRST = LAST - timedelta(days=365)
started = time.perf_counter()
year = ff.market.load(ROOT, "5m", FIRST, LAST, fidelity="1m", provider=PROVIDER).collect()
print(
    f"{year.height:,} five-minute bars of {ROOT} from {FIRST} to {LAST}, loaded in "
    f"{time.perf_counter() - started:.1f} s."
)
inconsistent = year.filter(pl.col("close") != pl.col("close_raw") + pl.col("adj_offset"))
print(f"Bars whose close is not close_raw + adj_offset: {inconsistent.height}.")
readable(year.tail(), ROOT)

## The rolls and the steps of adj_offset

Each roll inside that year: the trade date the new front contract starts, the contract it rolled from, and the step of `adj_offset` there.

Expected for MES: four rolls, in the week of each quarterly expiry. Each step is minus the roll's gap, the new contract's close minus the old one's in the last bar both traded in before the roll, so the adjusted prices run on without a jump while `close_raw` jumps by the gap.

In [ ]:
rolls = (
    year.with_columns(
        rolled_from=pl.col("contract").shift(1),
        step=pl.col("adj_offset") - pl.col("adj_offset").shift(1),
    )
    .filter("roll_flag")
    .select(
        "trade_date",
        "rolled_from",
        "contract",
        adj_offset=pl.col("adj_offset") / 1e9,
        step=pl.col("step") / 1e9,
    )
)
print(f"Rolls inside the window: {rolls.height}.")
rolls

## One real day, resampled

The last trade date with trades: each contract's 5-minute bars and then its daily bar, resampled from its own 1-minute bars, at the contract's own prices.

Expected: 276 five-minute bars for each contract on a full MES session, 17:00 to 16:00 Chicago, with the front contract, the one with the most volume, trading in nearly all of them. Its daily bar should match the same contract's full electronic session on your chart.

In [ ]:
traded = pl.col("volume") > 0
recent = ff.market.load(
    ROOT, "1d", LAST - timedelta(days=14), LAST, continuous=False, fidelity="1m", provider=PROVIDER
).collect()
DAY = recent.filter(traded).get_column("trade_date").max()
five_minutes = ff.market.load(
    ROOT, "5m", DAY, DAY, continuous=False, fidelity="1m", provider=PROVIDER
).collect()
print(f"Trade date {DAY}: each contract's 5-minute bars, then its daily bar.")
display(
    five_minutes.group_by("contract")
    .agg(bars=pl.len(), with_trades=traded.sum(), volume=pl.col("volume").sum())
    .sort("volume", descending=True)
)
readable(recent.filter((pl.col("trade_date") == DAY) & traded), ROOT)

## The comparison with the vendor's bars

That day's bars rebuilt from the finest data your store holds and compared with each timeframe stored, and its 1-minute bars resampled to 5-minute and then daily bars against the daily bars made from them at once, as `fefat data verify-bars` runs it.

Expected: `Differences: none.` With the vendor's 1-minute bars alone, as on this PC, the resampling is the comparison that runs; no 5-minute or daily bars are stored to compare with, and the comparisons from ticks are pending.

In [ ]:
report = verify_bars(store, ROOT, DAY, paths.cache_dir)
print(report.summary())

## 1-minute bars rebuilt from ticks

That day's 1-minute bars rebuilt from the product's trades and compared with the vendor's: the cell runs the day a trades file is in your store, and skips with a message until then.

Expected on this PC: the message that the cell skipped, since no trades file is imported yet.

In [ ]:
ticks = next(
    comparison
    for comparison in report.comparisons
    if comparison.provider == PROVIDER
    and [str(schema) for schema in comparison.route] == ["trades", "ohlcv-1m"]
)
if ticks.status is ComparisonStatus.COMPARED:
    print("\n".join(ticks.lines()))
else:
    print(f"Skipped until a trades file of {ROOT} is imported: {ticks.lines()[0]}")

## A dataset you imported through a preset

The last week of the first dataset that `fefat data import file` brought in through a preset, loaded contract by contract: an export can lack a contract, and a continuous series cannot roll across the hole.

Expected on this PC: MNQ from `ninjatrader`, the bars of MNQU2026 from 2026-09-11 to 2026-09-18; without such a dataset, the message that the cell skipped.

In [ ]:
preset_providers = [profile.provider for profile in preset_profiles().values()]
imported = listed.filter(pl.col("provider").is_in(preset_providers))
IMPORTED = None if imported.is_empty() else imported.row(0, named=True)
if IMPORTED is None:
    print(
        "Skipped: no dataset imported through a preset; bring in an export with "
        "`fefat data import file` (docs/data-guide.md)."
    )
else:
    end = IMPORTED["last_trade_date"]
    week = ff.market.load(
        IMPORTED["root"],
        "5m",
        end - timedelta(days=7),
        end,
        continuous=False,
        provider=IMPORTED["provider"],
    ).collect()
    contracts = ", ".join(week.get_column("contract").unique(maintain_order=True))
    print(
        f"{IMPORTED['provider']} {IMPORTED['root']} from {end - timedelta(days=7)} to {end}: "
        f"{week.height:,} five-minute bars of {contracts}."
    )
    display(readable(week.tail(), IMPORTED["root"]))

## Its quality report

That dataset's quality report, computed now as `fefat data quality` writes it: the summary, then the report.

Expected on this PC: flagged, with gaps on the sessions where the three MNQ contracts the import refused were the front month, as the coverage by month shows.

In [ ]:
if IMPORTED is None:
    print("Skipped: no dataset imported through a preset.")
else:
    quality = ff.market.assess_quality(
        store, IMPORTED["provider"], IMPORTED["root"], paths.cache_dir
    )
    print(quality.summary())
    display(Markdown(quality.markdown()))